# Clase 228 — Reproducibilidad: seeds, lock files, hashes y manifests

Solo stdlib + `numpy` + `pandas` + `scikit-learn`. Self-contained.

## 1. Seeds: con y sin

In [ ]:
import numpy as np

# Sin seed: distinto cada corrida
print('sin seed (1):', np.random.rand(5).round(4))
print('sin seed (2):', np.random.rand(5).round(4))

# Con seed: idéntico
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
print('con seed (a):', rng_a.random(5).round(4))
print('con seed (b):', rng_b.random(5).round(4))

## 2. `seed_everything()` helper

In [ ]:
import os, random

def seed_everything(seed: int = 42) -> None:
    """Seedea PRNGs Python + NumPy + entorno. Para Torch agregar:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True)
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)            # legacy global API
    return np.random.default_rng(seed)  # nuevo API: devolvemos el rng

rng = seed_everything(42)
print('random:', random.random())
print('numpy legacy:', np.random.rand(3).round(4))
print('numpy rng:   ', rng.random(3).round(4))

## 3. `PYTHONHASHSEED` y orden de `set`

OJO: `PYTHONHASHSEED` solo se respeta si se setea **antes** de iniciar el proceso. Setearla en runtime no cambia el hash de strings ya importados — la demo abajo es ilustrativa del problema.

In [ ]:
# Las strings tienen hash aleatorio por proceso si PYTHONHASHSEED no está fija
palabras = ['etica', 'fairness', 'privacidad', 'reproducibilidad', 'seeds']
s = set(palabras)
print('orden de iteración del set:', list(s))
print('hash("etica"):', hash('etica'))
print('\n→ En otra corrida del proceso, el orden y el hash pueden diferir')
print('  si PYTHONHASHSEED=random (default). Fix: PYTHONHASHSEED=0 python script.py')
print('  o, en código: sorted(s) en lugar de iterar el set.')

## 4. sklearn con `random_state` reproduce bit-a-bit

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

def fit_score(seed=42):
    m = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=1)
    m.fit(Xtr, ytr)
    return m.score(Xte, yte), m.feature_importances_[:3]

s1, fi1 = fit_score(42)
s2, fi2 = fit_score(42)
print(f'corrida 1: score={s1:.6f}, fi[:3]={fi1.round(6)}')
print(f'corrida 2: score={s2:.6f}, fi[:3]={fi2.round(6)}')
assert s1 == s2 and np.array_equal(fi1, fi2)
print('OK idéntico bit-a-bit con random_state fijo (en CPU, n_jobs=1)')

## 5. Float associativity: `sum` vs `np.sum` con float32

In [ ]:
rng = np.random.default_rng(0)
arr32 = rng.random(1_000_000, dtype=np.float32)

s_python = float(sum(arr32.tolist()))     # suma secuencial Python (float64)
s_numpy  = float(arr32.sum())              # suma vectorizada NumPy (float32 + pairwise)
s_sorted = float(np.sort(arr32).sum())     # suma en orden creciente

print(f'sum() Python:     {s_python:.6f}')
print(f'np.sum() default: {s_numpy:.6f}')
print(f'np.sum() sorted:  {s_sorted:.6f}')
print(f'\ndiff python vs numpy: {abs(s_python - s_numpy):.6f}')
print('→ Float no es asociativo. El orden de reducción importa.')
print('→ Por eso BLAS multi-thread y GPU reductions pueden diferir corrida a corrida.')

## 6. `sha256_of_df`: hash estable de un DataFrame

In [ ]:
import hashlib, pandas as pd

def sha256_of_df(df: pd.DataFrame) -> str:
    """Hash estable: orden de columnas alfabético, índice descartado, CSV utf-8."""
    normalized = (
        df.reset_index(drop=True)
          .sort_index(axis=1)        # columnas alfabéticas
    )
    payload = normalized.to_csv(index=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

df1 = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30], 'c': ['x', 'y', 'z']})
df2 = pd.DataFrame({'c': ['x', 'y', 'z'], 'b': [10, 20, 30], 'a': [1, 2, 3]})  # cols reordenadas
df3 = df1.copy(); df3.index = [99, 100, 101]                                    # índice raro
df4 = df1.copy(); df4.loc[0, 'a'] = 999                                         # contenido distinto

print('df1:', sha256_of_df(df1)[:16], '...')
print('df2:', sha256_of_df(df2)[:16], '... (cols reordenadas, mismo hash)')
print('df3:', sha256_of_df(df3)[:16], '... (índice raro, mismo hash)')
print('df4:', sha256_of_df(df4)[:16], '... (contenido cambia → hash distinto)')

assert sha256_of_df(df1) == sha256_of_df(df2) == sha256_of_df(df3)
assert sha256_of_df(df1) != sha256_of_df(df4)
print('\nOK hash sobrevive a reorden de columnas y reindex; cambia con el contenido')

## 7. Manifest del experimento

In [ ]:
import json, sys
from importlib.metadata import version, PackageNotFoundError

def pkg_version(name: str) -> str:
    try: return version(name)
    except PackageNotFoundError: return 'not-installed'

def build_manifest(df: pd.DataFrame, code_source: str, seed: int) -> dict:
    return {
        'data_hash': sha256_of_df(df),
        'code_hash': hashlib.sha256(code_source.encode('utf-8')).hexdigest(),
        'seed': seed,
        'python_version': sys.version.split()[0],
        'package_versions': {
            p: pkg_version(p) for p in ['numpy', 'pandas', 'scikit-learn']
        },
    }

df_train = pd.DataFrame({'x1': Xtr[:, 0], 'x2': Xtr[:, 1], 'y': ytr})
code = "model = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)"
manifest = build_manifest(df_train, code, seed=42)
print(json.dumps(manifest, indent=2))

## 8. Validación de reproducibilidad: el hash debe coincidir

In [ ]:
def assert_reproducible(df_now: pd.DataFrame, manifest_saved: dict) -> None:
    h_now = sha256_of_df(df_now)
    h_was = manifest_saved['data_hash']
    if h_now != h_was:
        raise RuntimeError(
            f'Data drift detectado.\n  manifest: {h_was[:16]}...\n  actual:   {h_now[:16]}...'
        )
    print(f'OK data_hash coincide: {h_now[:16]}...')

# Caso ok
assert_reproducible(df_train, manifest)

# Caso drift
df_drift = df_train.copy(); df_drift.loc[0, 'y'] = 1 - df_drift.loc[0, 'y']
try:
    assert_reproducible(df_drift, manifest)
except RuntimeError as e:
    print('detectado:', str(e).splitlines()[0])

## 9. Model card mínima (Mitchell et al. 2019)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def build_model_card(
    model, X_eval, y_eval, *,
    intended_use: str,
    training_data_hash: str,
    limitations: list,
    date_trained: str,   # ISO 8601, pasado por caller — no usamos datetime.now() para reproducibilidad
) -> dict:
    y_pred = model.predict(X_eval)
    return {
        'model_name': type(model).__name__,
        'model_params': model.get_params(),
        'intended_use': intended_use,
        'training_data_hash': training_data_hash,
        'metrics': {
            'accuracy': float(accuracy_score(y_eval, y_pred)),
            'f1': float(f1_score(y_eval, y_pred, average='macro')),
            'n_eval': int(len(y_eval)),
        },
        'limitations': limitations,
        'date_trained': date_trained,
    }

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1).fit(Xtr, ytr)
card = build_model_card(
    model, Xte, yte,
    intended_use='Demo educativa — clasificación binaria sintética. NO usar en producción.',
    training_data_hash=manifest['data_hash'],
    limitations=[
        'Dataset sintético sin validación externa',
        'No evaluado para fairness por subgrupo (ver Clase 224)',
        'Sin monitoreo de drift en serving (ver Parte 4 MLOps)',
    ],
    date_trained='2026-06-18',
)
print(json.dumps(card, indent=2, default=str))

## 10. Discusión: lock files, DVC vs Git LFS vs S3

**Lock files (`uv.lock`, `poetry.lock`, `Pipfile.lock`, `conda-lock`)** — pinean toda la **transitive tree** con hashes de wheels. `requirements.txt` con `pandas==2.2.0` no pinea `numpy`/`pytz`; otra máquina puede resolver versiones distintas. Para reproducibilidad real: lock file + `uv pip sync` (no `pip install -r`).

**Imagen Docker pineada por digest** — no `python:3.12-slim` (tag mutable) sino `python:3.12-slim@sha256:...` (digest inmutable). El tag puede repushearse; el digest no.

**Versionado de datasets**:
- **DVC** — workflow git-céntrico, pipelines declarativos con cache por inputs (`dvc repro`). Mejor para ML serio.
- **Git LFS** — transparente con git, pero sin pipelines ni cache. Bien para binarios chicos.
- **S3 versioning** — "versiones anteriores del objeto" sin metadata. Bien si ya vivís en AWS y no necesitás más.
- **lakeFS / Quilt / Pachyderm** — branching real sobre data lakes; enterprise scale.

**Pineau et al. 2021** propone el NeurIPS Reproducibility Checklist: reportar **media ± std de 3+ seeds**, no un seed mágico; especificar hardware, librería, versión; publicar código + datos + hiperparámetros.

## Ejercicio guiado

1. Tomá un dataset real (por ejemplo Iris vía `sklearn.datasets`). Computá su `dataset_hash` y guardá `manifest.json`.
2. Modificá un valor del dataset (1 fila) y verificá que `assert_reproducible` falla.
3. Generá un `requirements.lock` con `uv pip compile requirements.in -o requirements.lock` (instalá `uv` con `pip install uv`).
4. Escribí un `Dockerfile` que use `python:3.12-slim@sha256:<digest>` (mirá el digest en Docker Hub).
5. Inicializá un repo DVC (`dvc init`) y trackeá el dataset con `dvc add data/dataset.csv`. Inspeccioná el `.dvc` pointer generado.

## Conclusiones

- **Seeds**: `seed_everything()` cubre `random`, `numpy`, `PYTHONHASHSEED`; sumar `torch.*` y `CUBLAS_WORKSPACE_CONFIG` si hay GPU.
- **Lock files** > `requirements.txt` — pinean toda la transitive tree con hashes.
- **`sha256_of_df`** estable (cols ordenadas, sin índice) = `dataset_id` reproducible.
- **Manifest + model card** son los dos artefactos que acompañan a TODO experimento serio.
- Reportá **media ± std de varios seeds**, no un seed mágico (NeurIPS Reproducibility Checklist).